In [ ]:
# Imports
import os
import shutil
from pathlib import Path
from PIL import Image


In [ ]:
# Data mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.listdir('/content/drive/MyDrive')

In [ ]:
# I'll be using the YOLO v8 model from Ultralytics (probably nano, but maybe bigger)
!pip install ultralytics
import ultralytics
ultralytics.checks()


In [ ]:
# Copy from Drive to local Colab disk (better I/O)
DRIVE_PATH = "/content/drive/MyDrive/SkiTB_data"
LOCAL_PATH = "/content/skitb"

# Resetting local copy
if os.path.exists('/content/skitb'):
    shutil.rmtree('/content/skitb')

print("Copying zip...")
shutil.copy('/content/drive/MyDrive/SkiTB_data/SkiTB_subset.7z', '/content/')
print("Copy done!")

print("Unzipping...")
!7z x /content/SkiTB_subset.7z -o/content/skitb
print("Done!")

print(os.listdir('/content/skitb'))

In [ ]:
# Check dimensions of a sample frame
sample = "/content/skitb/SkiTB/FS/FS0020/frames/00300.jpg"
img = Image.open(sample)
print(img.size)  # (width, height), (1280, 720) in our case


In [ ]:
# Converting images from dataset to YOLO compliant shape
from pathlib import Path

def convert_skitb_to_yolo(sequence_dir, output_dir, img_width=1280, img_height=720, visibility_only=True):
    sequence_dir = Path(sequence_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Work at the SC (single camera) clip level
    sc_dir = sequence_dir / "SC"
    frames_dir = sequence_dir / "frames"

    # Check each single cam
    for clip_dir in sc_dir.iterdir():
        if not clip_dir.is_dir():
            continue

        boxes = (clip_dir / "boxes.txt").read_text().strip().splitlines()
        visibilities = (clip_dir / "visibilities.txt").read_text().strip().splitlines()
        frame_indices = (clip_dir / "frames.txt").read_text().strip().splitlines()

        for i, (box, vis, frame_idx) in enumerate(zip(boxes, visibilities, frame_indices)):

            # Skip occluded frames if visibility_only=True
            # (May change later but most jumps will have most of the
            # rider in frame)
            if visibility_only and vis.strip() == "0":
                continue

            x, y, w, h = map(int, box.strip().split(","))

            # Convert to YOLO normalized center format
            cx = (x + w / 2) / img_width
            cy = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            # Clamp to [0, 1] in case of any annotation edges exceeding frame
            cx = max(0, min(1, cx))
            cy = max(0, min(1, cy))
            w_norm = max(0, min(1, w_norm))
            h_norm = max(0, min(1, h_norm))

            # Label file named to match the frame image
            frame_name = frame_idx.strip().zfill(5)  # e.g. "00027"
            label_path = output_dir / f"{frame_name}.txt"

            # class 0 = person/skier
            with open(label_path, "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# Run it over all sequences (recorded runs)
base_dir = "/content/skitb/SkiTB/FS"
output_base = "/content/skitb/yolo/labels"

for seq in sorted(os.listdir(base_dir)):
    seq_path = os.path.join(base_dir, seq)
    if not os.path.isdir(seq_path):
        continue

    out_path = os.path.join(output_base, seq)
    convert_skitb_to_yolo(seq_path, out_path, img_width=1920, img_height=1080)
    print(f"Converted {seq}")

In [ ]:
# Sanity check that everything looks right

# Check label folders were created
print(os.listdir('/content/skitb/yolo/labels'))

# Peek at one random label file
sample_label = list(Path('/content/skitb/yolo/labels').rglob('*.txt'))[0]
print(f"\nSample file: {sample_label}")
print(sample_label.read_text()[:200])

# Count total label files generated
all_labels = list(Path('/content/skitb/yolo/labels').rglob('*.txt'))
print(f"\nTotal label files: {len(all_labels)}")


In [ ]:
print(os.listdir('/content/skitb/SkiTB/FS'))